# 🐧 Level 2 – Linux Explorer Lab (for WSL Ubuntu)

> **Read this first:**  
> All exercises are run directly in your WSL terminal. Type every command yourself and compare the outputs.  
> If you see `$`, it’s the shell prompt – do not type it.


## ⚙️ 0. Lab Setup

Install missing tools (run once):

```bash
sudo apt update
sudo apt install -y htop ncdu locate p7zip-full tree
```

Then build the `locate` database so the `locate` command works:

```bash
sudo updatedb
```

This command takes time wait 5 min

Now everything is ready.


## 🔍 1. Searching Files & Directories (Hands‑On)

### 1.1 – Create a test directory with dummy files

```bash
mkdir -p ~/lab-search/{subdir1,subdir2}
cd ~/lab-search
touch file1.txt file2.log file3.sh image.png
touch subdir1/notes.txt subdir2/backup.tar.gz
touch .hiddenfile
```

### 1.2 – Find files by name

```bash
find . -name "*.txt"
```

Expected output:

```
./file1.txt
./subdir1/notes.txt
```

### 1.3 – Find only directories

```bash
find . -type d
```

Output includes `.`, `./subdir1`, `./subdir2`.

### 1.4 – Find files modified in the last 5 minutes (just after creating them)

```bash
find . -mmin -5 -type f
```

All the files you just created should appear.

### 1.5 – Find by size (create a larger file)

```bash
dd if=/dev/zero of=bigfile bs=1M count=50 2>/dev/null
find . -size +10M -type f -printf "%p %s bytes\n"
```

You’ll see `./bigfile` with ~50 MB.

### 1.6 – Find and delete empty files (simulate)

```bash
touch empty1 empty2
find . -empty -type f -delete
ls empty1 empty2   # should say “No such file or directory”
```

### 1.7 – Grep for text in files

```bash
echo "error: disk full" > app.log
echo "info: starting" >> app.log
grep "error" app.log
```

Output:

```
error: disk full
```

### 1.8 – Recursive grep with line numbers

```bash
grep -rn "error" .
```

It will show `./app.log:1:error: disk full` (and possibly other matches).

### 1.9 – Case‑insensitive search

```bash
echo "Error: timeout" >> app.log
grep -i "error" app.log
```

Both “error” and “Error” appear.

### 1.10 – Locate a system file (fast database lookup)

```bash
locate hosts | head -5
```

You’ll see paths like `/etc/hosts`, `/etc/hosts.allow`, etc.

### 1.11 – Which and whereis

```bash
which bash
whereis bash
```

Output:

```
/usr/bin/bash
bash: /usr/bin/bash /usr/share/man/man1/bash.1.gz
```


### 🧠 Quick wisdom

- `find` is slow but versatile; use `locate` for known filenames (after `sudo updatedb`).
- Use `-exec` only after testing with `-print`.
- `grep -rn "pattern"` is your most common content search.

**Try it yourself:** In `~/lab-search`, find all files that have a `.sh` extension and change their permission to executable.

<details>
<summary>Solution</summary>

```bash
find ~/lab-search -name "*.sh" -type f -exec chmod +x {} \;
```

</details>


## ✂️ 2. Text Processing & Transformation (Hands‑On)

We'll create sample data to process.

### 2.1 – Prepare a fake log file

```bash
cat > /tmp/access.log << 'EOF'
192.168.1.10 - - [08/Jun/2026:10:15:32] "GET /index.html HTTP/1.1" 200 1024
10.0.0.5 - - [08/Jun/2026:10:15:33] "POST /login HTTP/1.1" 401 512
192.168.1.10 - - [08/Jun/2026:10:15:34] "GET /style.css HTTP/1.1" 200 256
10.0.0.5 - - [08/Jun/2026:10:15:35] "GET /admin HTTP/1.1" 403 128
172.16.0.1 - - [08/Jun/2026:10:15:36] "GET /index.html HTTP/1.1" 200 1024
EOF
```

### 2.2 – Extract the first column (IP address)

```bash
cut -d ' ' -f1 /tmp/access.log
```

Output:

```
192.168.1.10
10.0.0.5
192.168.1.10
10.0.0.5
172.16.0.1
```

### 2.3 – Count unique IPs

```bash
cut -d ' ' -f1 /tmp/access.log | sort | uniq -c | sort -rn
```

Output:

```
      2 192.168.1.10
      2 10.0.0.5
      1 172.16.0.1
```

### 2.4 – Convert to uppercase

```bash
echo "hello world" | tr 'a-z' 'A-Z'
```

Output: `HELLO WORLD`

### 2.5 – Remove duplicate spaces (squeeze)

```bash
echo "too    many   spaces" | tr -s ' '
```

Output: `too many spaces`

### 2.6 – Count lines, words, characters

```bash
wc /tmp/access.log
```

You'll see `5 45 292 /tmp/access.log` (numbers may differ slightly).

### 2.7 – Sort numerically (create number list)

```bash
echo -e "10\n2\n15\n3" | sort -n
```

Output:

```
2
3
10
15
```

### 2.8 – Use awk to print specific fields

```bash
awk '{print $1, $9, $10}' /tmp/access.log
```

Shows IP, status code, and bytes.

### 2.9 – awk with condition (show only 200 responses)

```bash
awk '$9 == 200 {print $1, $7}' /tmp/access.log
```

Output:

```
192.168.1.10 /index.html
192.168.1.10 /style.css
172.16.0.1 /index.html
```

### 2.10 – sed find-and-replace

```bash
sed 's/192.168.1/10.10.10/g' /tmp/access.log
```

All `192.168.1` become `10.10.10`.

### 2.11 – Delete lines containing “401”

```bash
sed '/401/d' /tmp/access.log
```

That line disappears.

### 2.12 – Print only lines 2 to 4

```bash
sed -n '2,4p' /tmp/access.log
```


### 🧠 Pipeline wisdom

- Always `sort` before `uniq`.
- `awk` is great for column math; `sed` for text substitution.
- Use `tr` for character‑level changes, not string replacements.

**Try it yourself:** From `/etc/passwd`, extract all usernames and shell, then show only those using `/bin/bash`.

<details>
<summary>Solution</summary>

```bash
cut -d: -f1,7 /etc/passwd | grep "/bin/bash"
```

</details>


## ⚡ 3. Process Management & Monitoring (Hands‑On)

We'll use `sleep` commands as example processes.

### 3.1 – Start a long‑running process in the background

```bash
sleep 500 &
```

You’ll see a job number and PID, like `[1] 12345`.

### 3.2 – List all processes, filter for sleep

```bash
ps aux | grep sleep | grep -v grep
```

You’ll see the `sleep 500` process.

### 3.3 – Get PID with pgrep

```bash
pgrep sleep
```

Returns the PID directly.

### 3.4 – Check process tree

```bash
ps auxf | grep sleep
```

Shows the tree (bash → sleep).

### 3.5 – Use top in batch mode (one snapshot)

```bash
top -b -n 1 | head -15
```

You’ll see a static system summary.

### 3.6 – Use htop (interactive)

```bash
htop
```

Press `q` to quit. (Just a quick look.)

### 3.7 – Kill the sleep process (first find PID)

```bash
kill $(pgrep sleep)
```

Wait a moment, then check again with `ps aux | grep sleep` – it should be gone.

### 3.8 – Start another, then send STOP and CONT

```bash
sleep 500 &
# Note PID, let's say 12346
kill -STOP 12346
ps aux | grep sleep   # shows "T" (stopped)
kill -CONT 12346
ps aux | grep sleep   # now "S" (sleeping) again
kill 12346            # terminate
```

### 3.9 – Change priority with nice and renice

```bash
nice -n 10 sleep 100 &
renice -n 15 -p $(pgrep sleep)
```

Check with `ps -l -p $(pgrep sleep)`; the NI column shows the niceness.

### 3.10 – Show processes using a port (maybe SSH)

```bash
sudo ss -tlnp | grep 22
```

You’ll see `sshd` listening.

### 3.11 – Run a job in background, then bring to foreground

```bash
sleep 60
# Press Ctrl+Z immediately
bg
jobs
fg 1
```


### 🧠 Process safety

- Never `kill -9` first; try SIGTERM (default) and wait.
- `pkill -f` matches full command line – be specific.
- `watch -n 1 'ps aux | grep thing'` is a quick monitor.

**Try it yourself:** Start two `sleep` processes, list their PIDs, then kill only the one with higher PID.

<details>
<summary>Solution</summary>

```bash
sleep 200 &
sleep 300 &
pids=($(pgrep sleep))
if [ ${pids[1]} -gt ${pids[0]} ]; then
    kill ${pids[1]}
else
    kill ${pids[0]}
fi
```

</details>


## 📦 4. Archiving & Compression (Hands‑On)

We'll create files to compress.

### 4.1 – Create a test directory with content

```bash
mkdir -p ~/lab-archive/docs
echo "important data" > ~/lab-archive/docs/file1.txt
echo "more data" > ~/lab-archive/docs/file2.txt
dd if=/dev/zero of=~/lab-archive/docs/big.dat bs=1M count=10 2>/dev/null
```

### 4.2 – Create a tar archive (uncompressed)

```bash
cd ~/lab-archive
tar -cvf backup.tar docs/
```

Output shows the files added.

### 4.3 – Create a gzip‑compressed tar (most common)

```bash
tar -czvf backup.tar.gz docs/
```

Note the size difference:

```bash
ls -lh backup.tar backup.tar.gz
```

### 4.4 – List contents without extracting

```bash
tar -tzvf backup.tar.gz
```

### 4.5 – Extract to a new location

```bash
mkdir extracted
tar -xzvf backup.tar.gz -C extracted
ls -R extracted
```

### 4.6 – Compress a single file with gzip

```bash
gzip -k docs/file1.txt
ls -l docs/file1.txt.gz
```

`-k` keeps the original.

### 4.7 – Decompress a .gz file

```bash
gunzip docs/file1.txt.gz
```

### 4.8 – Use bzip2 and xz (compare sizes)

```bash
cp docs/big.dat big.dat
gzip -9 -k big.dat
bzip2 -9 -k big.dat
xz -9 -k big.dat
ls -lh big.dat*
```

### 4.9 – Stream a tar over a “pipe” (simulate)

```bash
tar -czf - docs/ | tar -tzf -
```

This shows the contents without writing to disk.

### 4.10 – Zip archive (cross‑platform)

```bash
zip -r archive.zip docs/
unzip -l archive.zip
```

### 4.11 – 7‑Zip (7z) for maximum compression

```bash
7z a -mx=9 archive.7z docs/
7z l archive.7z
```


### 🧠 Compression tips

- `tar -czvf` is your everyday backup friend.
- `--exclude='*.log'` keeps logs out of archives.
- Always test extraction (`tar -tzf`) before trusting.
- `xz -9` is overkill for quick backups; use gzip.

**Try it yourself:** Create a tar.gz of `/etc` (with root permissions), but exclude everything under `/etc/ssl`.

<details>
<summary>Solution</summary>

```bash
sudo tar -czvf etc-backup.tar.gz /etc --exclude='etc/ssl/*'
```

</details>


## 👥 5. User & Group Management (Hands‑On)

_Note: Some commands require `sudo`. WSL might have limited user management, but you can practice commands that work locally._

### 5.1 – See who you are

```bash
whoami
id
```

Output shows your username, uid, gid, and groups.

### 5.2 – List all users from the system database

```bash
getent passwd | tail -10
```

### 5.3 – List all groups

```bash
getent group | tail -10
```

### 5.4 – Create a new user (if possible; otherwise simulate with `useradd -D`)

```bash
sudo useradd -m -s /bin/bash testuser
# If fails due to WSL, just read the man page
```

### 5.5 – Set password (not practical in WSL, but you can see the command)

```bash
sudo passwd testuser   # would prompt for password
```

### 5.6 – Add user to supplementary group

```bash
sudo usermod -aG sudo testuser   # adds to sudo group
```

### 5.7 – Check groups of a user

```bash
id testuser
```

### 5.8 – Lock and unlock account

```bash
sudo usermod -L testuser
sudo usermod -U testuser
```

### 5.9 – Delete a user (cleanup)

```bash
sudo userdel -r testuser
```

### 5.10 – Edit sudoers safely

```bash
sudo visudo   # (opens editor; just quit without changes)
```

### 5.11 – Switch user (if testuser exists)

```bash
su - testuser -c 'whoami'
```

### 5.12 – Check sudo permissions

```bash
sudo -l
```


### 🧠 User management wisdom

- `visudo` is mandatory – it checks syntax before saving.
- Use `usermod -aG` to avoid overwriting group list.
- `getent` respects LDAP and other name services, unlike reading `/etc/passwd`.

**Try it yourself:** Create a group called `devops`, then check its GID.

<details>
<summary>Solution</summary>

```bash
sudo groupadd devops
getent group devops
```

</details>


## 💾 6. Disk & Filesystem Management (Hands‑On)

### 6.1 – Check disk usage

```bash
df -h
```

Shows all mounted filesystems with human‑readable sizes.

### 6.2 – Show inode usage

```bash
df -i
```

### 6.3 – Find large directories

```bash
du -sh /var/* | sort -h | tail -10
```

### 6.4 – Interactive disk usage (ncdu)

```bash
ncdu /var/log
```

Navigate with arrow keys, press `?` for help, `q` to quit.

### 6.5 – List block devices

```bash
lsblk
```

You’ll see your disk layout (sda, partitions, etc.).

### 6.6 – Get UUID of the root partition

```bash
blkid /dev/sda1   # or the partition that shows "/" in lsblk
```

Note the UUID.

### 6.7 – View /etc/fstab (persistent mounts)

```bash
cat /etc/fstab
```

### 6.8 – Create a test file system (loop device simulation)

```bash
dd if=/dev/zero of=/tmp/test.img bs=1M count=100
sudo mkfs.ext4 /tmp/test.img
```

(We skip mounting since it requires more setup, but you can see the created filesystem.)

### 6.9 – Check and repair (fsck) – just read‑only check

```bash
sudo fsck -n /tmp/test.img
```

### 6.10 – Find the 10 largest files in your home

```bash
find ~ -type f -exec du -h {} + 2>/dev/null | sort -rh | head -10
```


### 🧠 Disk wisdom

- `df -i` warns you about inode exhaustion.
- `ncdu` is the fastest way to find space hogs.
- Always test fstab with `mount -a` before reboot.

**Try it yourself:** Find the partition where `/var/log` is mounted using `df`.

<details>
<summary>Solution</summary>

```bash
df /var/log
```

</details>


## 📜 7. Bash Intermediate – Level Up Your Scripts (Hands‑On)

We’ll write small scripts and run them immediately.

### 7.1 – Script with arrays and loops

```bash
cat > ~/lab-script-array.sh << 'EOF'
#!/bin/bash
servers=("web01" "db01" "cache01")
for s in "${servers[@]}"; do
    echo "Pinging $s..."
    # simulate
    sleep 0.1
done
echo "All done."
EOF
chmod +x ~/lab-script-array.sh
./lab-script-array.sh
```

### 7.2 – Function with return code

```bash
cat > ~/lab-script-func.sh << 'EOF'
#!/bin/bash
check_file() {
    if [ -f "$1" ]; then
        echo "OK: $1 exists"
        return 0
    else
        echo "FAIL: $1 missing" >&2
        return 1
    fi
}

check_file /etc/hosts
echo "Return code: $?"
check_file /nonexistent
echo "Return code: $?"
EOF
chmod +x ~/lab-script-func.sh
./lab-script-func.sh
```

### 7.3 – Debug mode on/off

```bash
cat > ~/lab-script-debug.sh << 'EOF'
#!/bin/bash
set -euo pipefail

echo "Normal operation"
set -x
echo "Debugging on"
date
set +x
echo "Debugging off"
EOF
chmod +x ~/lab-script-debug.sh
./lab-script-debug.sh
```

Observe lines prefixed with `+` when `set -x` is active.

### 7.4 – Associative arrays (Bash 4+)

```bash
cat > ~/lab-script-assoc.sh << 'EOF'
#!/bin/bash
declare -A config
config[host]="db1.prod"
config[port]="5432"
config[user]="admin"

echo "Connecting to ${config[host]}:${config[port]} as ${config[user]}"
EOF
chmod +x ~/lab-script-assoc.sh
./lab-script-assoc.sh
```

### 7.5 – Parameter expansion tricks

```bash
filename="backup-2026-06-08.tar.gz"
echo "Original: $filename"
echo "Base name: ${filename%.*.*}"
echo "Extension: ${filename##*.}"
echo "No dash: ${filename//-/}"
echo "First 6 chars: ${filename:0:6}"
echo "Length: ${#filename}"
```

### 7.6 – Pipefail demonstration

```bash
cat > /tmp/pipefail-test.sh << 'EOF'
#!/bin/bash
set -o pipefail
false | true
echo "Exit status: $?"
EOF
bash /tmp/pipefail-test.sh
```

Without `pipefail`, the pipeline would succeed (exit code of `true`). With it, it fails.


### 🧠 Scripting discipline

- Always start with `set -euo pipefail`.
- Quote your arrays and variables.
- Use `"${var}"` to avoid splitting.
- Functions should `return` exit codes, `echo` output.

**Try it yourself:** Write a script that takes a filename as argument and prints its size if it exists, otherwise an error.

<details>
<summary>Solution</summary>

```bash
#!/bin/bash
[ -z "$1" ] && { echo "Usage: $0 <file>"; exit 1; }
if [ -f "$1" ]; then
    du -h "$1" | cut -f1
else
    echo "File not found" >&2
    exit 2
fi
```

</details>


## 🧪 Final Challenge – Log Analyzer

You are given a simulated access log `/tmp/final-log.txt`:

```bash
cat > /tmp/final-log.txt << 'EOF'
203.0.113.5 - - [08/Jun/2026:12:00:01] "GET /api/data HTTP/1.1" 200 512
198.51.100.22 - - [08/Jun/2026:12:00:02] "POST /login HTTP/1.1" 401 128
203.0.113.5 - - [08/Jun/2026:12:00:03] "GET /api/status HTTP/1.1" 500 64
203.0.113.5 - - [08/Jun/2026:12:00:04] "GET /home HTTP/1.1" 200 1024
198.51.100.22 - - [08/Jun/2026:12:00:05] "GET /api/data HTTP/1.1" 200 512
EOF
```

Write a bash script `/tmp/log-report.sh` that:

1. Prints total number of requests.
2. Lists unique IPs with request count.
3. Shows only error lines (status ≥ 400).
4. Saves the result to `/tmp/report.txt`.

Make the script executable and run it.

<details>
<summary>Solution</summary>

```bash
cat > /tmp/log-report.sh << 'SCRIPT'
#!/bin/bash
set -euo pipefail
LOG="/tmp/final-log.txt"
REPORT="/tmp/report.txt"

echo "--- Log Report ---" > "$REPORT"
echo "Total requests: $(wc -l < "$LOG")" >> "$REPORT"
echo "" >> "$REPORT"
echo "Requests per IP:" >> "$REPORT"
awk '{print $1}' "$LOG" | sort | uniq -c | sort -rn >> "$REPORT"
echo "" >> "$REPORT"
echo "Errors (status >= 400):" >> "$REPORT"
awk '$9 >= 400' "$LOG" >> "$REPORT"
cat "$REPORT"
SCRIPT
chmod +x /tmp/log-report.sh
/tmp/log-report.sh
```

</details>
